# 01_explore_data — xem dữ liệu đã xử lý (tuần 1)
Đọc Dataset `mamba-btl-data` do `data/prepare.py` tạo: thống kê các tập, tách từng bài theo ký tự `<doc>`, đọc nội dung bài, từ vựng, tập mô phỏng gõ phím, phân bố độ dài bài **sau lọc** (bổ sung cho chương Dữ liệu, issue #25).

**Kaggle:** Accelerator **None** (CPU là đủ) · Internet On · Add Input Dataset `mamba-btl-data`.
**Máy cá nhân:** tải Dataset về, giải nén vào `data/processed/` của repo, mở notebook này từ thư mục gốc repo (bỏ qua cell clone).

In [ ]:
# 1. Clone repo (chỉ trên Kaggle; trên máy cá nhân bỏ qua cell này)
import os
if os.path.exists("/kaggle/working"):
    %cd /kaggle/working
    !rm -rf mamba-vi-keyboard && git clone -q https://github.com/daisyngocha/mamba-vi-keyboard.git
    %cd /kaggle/working/mamba-vi-keyboard
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

In [ ]:
# 2. Nạp từ vựng và thống kê các tập
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from data.loader import CharVocab, DOC_ID, UNK_ID, resolve_data_dir

D = resolve_data_dir(None)          # Kaggle: /kaggle/input/.../mamba-btl-data · máy cá nhân: data/processed
assert (D / "vocab.json").exists(), f"Không thấy dữ liệu ở {D}"
vocab = CharVocab.load(D / "vocab.json")
meta = json.loads((D / "meta.json").read_text(encoding="utf-8"))
print("Thư mục dữ liệu:", D)
print("vocab_size:", meta["vocab_size"], "| nguồn:", meta["dataset"], "| seed:", meta["seed"])
pd.DataFrame({k: v for k, v in meta["splits"].items() if k != "sample"}).T

In [ ]:
# 3. Tách từng bài theo <doc> (id 1): độ dài, số <unk>, đoạn mở đầu
def load_docs(split):
    arr = np.memmap(D / f"{split}.bin", dtype=np.uint16, mode="r")
    starts = np.flatnonzero(arr == DOC_ID)
    ends = np.append(starts[1:], len(arr))
    unk_pos = np.flatnonzero(arr == UNK_ID)
    docs = pd.DataFrame({"start": starts, "len": ends - starts - 1})   # không tính ký tự <doc>
    docs["unk"] = np.searchsorted(unk_pos, ends) - np.searchsorted(unk_pos, starts)
    return arr, docs

SPLIT = "val"                        # "train" | "val" | "test"
arr, docs = load_docs(SPLIT)
docs["mở đầu"] = [vocab.decode(arr[s + 1:s + 120]).replace("\n", " ⏎ ") for s in docs.start]
print(f"{SPLIT}: {len(docs)} bài, {len(arr):,} ký tự, độ dài trung vị {int(docs.len.median()):,} ký tự")
docs.head(20)

In [ ]:
# 4. Đọc trọn một bài (đổi i)
i = 0
s, n = docs.start[i], docs.len[i]
print(f"Bài {i}: {n:,} ký tự, {docs.unk[i]} ký tự <unk>\n")
print(vocab.decode(arr[s + 1:s + 1 + n])[:3000])

In [ ]:
# 5. Phân bố độ dài bài SAU LỌC trên cả 3 tập (hình cho chương Dữ liệu)
lens = {sp: load_docs(sp)[1]["len"] for sp in ("train", "val", "test")}
summary = pd.DataFrame({sp: l.describe(percentiles=[.1, .5, .9]) for sp, l in lens.items()}).T
display(summary.round(0))
all_len = pd.concat(lens.values())
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(np.log10(all_len), bins=50)
ax.axvline(np.log10(2000), color="red", ls="--", label="ngưỡng lọc 2.000 ký tự")
ax.set(xlabel="log10(độ dài bài, ký tự)", ylabel="số bài", title=f"Độ dài {len(all_len):,} bài giữ lại (train + val + test)")
ax.legend(); fig.tight_layout()
out = Path("/kaggle/working" if os.path.exists("/kaggle/working") else ".") / "length_hist_kept.png"
fig.savefig(out, dpi=150); print("Đã lưu", out)

In [ ]:
# 6. Từ vựng: 273 ký tự giữ lại + tần suất trên tập train
train = np.memmap(D / "train.bin", dtype=np.uint16, mode="r")
cnt = np.bincount(train, minlength=len(vocab))
voc = pd.DataFrame({"id": range(len(vocab)), "ký tự": [repr(c) if c.isspace() else c for c in vocab.itos], "số lần": cnt})
voc["tỷ lệ %"] = (100 * voc["số lần"] / cnt.sum()).round(4)
print("".join(c for c in vocab.itos[2:] if not c.isspace()))
print("Chữ hoa có dấu trong từ vựng:", "".join(c for c in vocab.itos[2:] if c.isupper() and not c.isascii()))
voc.sort_values("số lần", ascending=False).head(30)

In [ ]:
# 7. Tập mô phỏng gõ phím (2.000 vị trí trong test)
sim = pd.read_json(D / "typing_sim.jsonl", lines=True)
print(sim["mode"].value_counts().to_dict(), "| độ dài typed trung vị:", int(sim.typed.str.len().median()))
sim["typed (50 ký tự cuối)"] = sim.typed.str[-50:].str.replace("\n", " ⏎ ")
sim[["mode", "typed (50 ký tự cuối)", "prefix", "target"]].sample(15, random_state=0)